# Staffing Model (2025)

The model turns a 2025 monthly demand forecast into a month-by-month crew plan for the four in-scope departments using the team's **staffing policy**, and evaluates its cost and on-time performance.

| Section | What it does |
|---|---|
| 1. Parameters | Crew capacity, cost and starting crews from the brief (ASSUMED, used as stated) |
| 2. Demand input | 2025 forecast |
| 3. Modeling assumptions | Rules the brief leaves open, written down explicitly |
| 4. Plan evaluator | Simulates a crew plan month by month and returns costs |
| 5. Plan validation | Checks a plan obeys hiring, release and seasonal rules |
| 6. Runners | Apply the policy to all departments, with or without a different actual demand |
| 7. Staffing policy | The team's decision rules |
| 8. 2025 staffing plan | Crews by month and the plan's cost and on-time results |
| 9. Sensitivity | Demand ±20% |
| 10. Lookup table | Lookup table for the intake pipeline |
| 11. Process Intake Pipeline | Lookup table for the intake pipeline |



## 0. Setup

In [45]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

MONTHS = pd.period_range("2025-01", "2025-12", freq="M")
DAYS_PER_MONTH = 21

## 1. Parameters (ASSUMED, from the brief)

These values are planning assumptions given in the brief and are used exactly as stated.

| Dept | Code | Daily capacity / crew | Annual cost / crew | Crews, Jan 2025 |
|---|---|---|---|---|
| Public Works | PWDx | 12 | \$130,000 | 50 |
| Transportation | BTDT | 15 | \$110,000 | 18 |
| Parks & Recreation | PARK | 6 | \$120,000 | 10 |
| Inspectional Services | ISD | 4 | \$140,000 | 16 |

Lifecycle and cost rules: 21 working days per month; new crews need 1 month of hiring and training; releasing a crew needs 1 month of notice and costs 25% of its annual cost; Parks may add seasonal crews from April to October at 1.3× the monthly cost, with no training delay and no release cost. Each overdue request costs \$20, plus \$100 if still open 30 days after its target date.

In [46]:
PARAMS = {
    "PWDx": {"name": "Public Works",          "daily_cap": 12, "annual_cost": 130_000, "start_crews": 50},
    "BTDT": {"name": "Transportation",        "daily_cap": 15, "annual_cost": 110_000, "start_crews": 18},
    "PARK": {"name": "Parks & Recreation",    "daily_cap": 6,  "annual_cost": 120_000, "start_crews": 10},
    "ISD":  {"name": "Inspectional Services", "daily_cap": 4,  "annual_cost": 140_000, "start_crews": 16},
}

RELEASE_COST_RATE = 0.25        # of annual cost per released crew
SEASONAL_MULTIPLIER = 1.3       # Parks seasonal crew cost vs. regular monthly cost
SEASONAL_MONTHS = range(4, 11)  # April..October
SEASONAL_DEPTS = {"PARK"}
OVERDUE_COST = 20
OVERDUE_30_EXTRA = 100

for code_, p in PARAMS.items():
    p["monthly_cost"] = p["annual_cost"] / 12
    p["monthly_cap_per_crew"] = p["daily_cap"] * DAYS_PER_MONTH

pd.DataFrame(PARAMS).T[["name", "daily_cap", "monthly_cap_per_crew", "annual_cost", "monthly_cost", "start_crews"]]

,name,daily_cap,monthly_cap_per_crew,annual_cost,monthly_cost,start_crews
PWDx,Public Works,12,252,130000,10833.333333,50
BTDT,Transportation,15,315,110000,9166.666667,18
PARK,Parks & Recreation,6,126,120000,10000.0,10
ISD,Inspectional Services,4,84,140000,11666.666667,16


## 2. Input — 2025 forecast

In [47]:
demand_df = (
    pd.read_csv("monthly_forecast_2025.csv")
    .assign(month=lambda d: pd.to_datetime(d["month"]).dt.to_period("M"))
    .rename(columns={"forecast_requests": "requests"})
    [["department", "month", "requests"]]
)
demand_df.pivot(index="month", columns="department", values="requests")

department,BTDT,ISD,PARK,PWDx
month,,,,
2025-01,6039,1459,919,12437
2025-02,6117,1501,785,10285
2025-03,7584,1336,1219,11995
2025-04,7155,1370,1486,12740
2025-05,7729,1530,2173,13031
2025-06,7481,1801,2829,12673
2025-07,6685,1762,2662,13678
2025-08,7090,1997,2360,16474
2025-09,7990,1804,1658,15662


## 3. Modeling assumptions

The brief leaves some rules open. These are the choices this model makes; each one is a parameter or a clearly marked line in the evaluator, so the team can change it and re-run.

1. **No backlog on Jan 1, 2025** (given in the brief).
2. **First in, first out.** Each month, crews close the oldest open requests first.
3. **Target date = end of the month the request was opened** (`TARGET_MONTHS = 1`). A request still open at month-end becomes overdue (\$20, counted once). A request still open one more month later is treated as 30+ days past target (+\$100, counted once). *This is a deliberate simplification applied to all departments: many real requests have longer targets, so overdue counts and costs here are conservative.*
4. **Hiring.** A crew active in month *m* was hired in *m−1*. During that training month it is paid but closes nothing (`PAY_DURING_TRAINING = True`).
5. **Release.** A crew removed in month *m* was given notice in *m−1* and works through *m−1*. The 25% release cost is charged in month *m*.
6. **January is fixed.** Changing January crews would need a decision in December 2024, so January uses the starting crews from the brief.
7. **Parks seasonal crews** start and stop instantly, only April–October, with no release cost.

In [48]:
TARGET_MONTHS = 1
PAY_DURING_TRAINING = True

## 4. Plan evaluator

`simulate(dept, crews, demand, seasonal=None)` takes a plan and returns a month-by-month table.

- `crews`: list of 12 integers, regular crews **active** in each month.
- `seasonal`: list of 12 integers, Parks seasonal crews (zeros for other departments).
- `demand`: list of 12 forecast request counts.

Monthly total cost follows the structure the team derived:

**crew cost (active + in training) + seasonal cost + release cost + overdue cost**

Idle crew cost is reported as a diagnostic only. It is already inside crew cost, so it is **not** added again.

In [49]:
def derive_moves(crews):
    """Hires starting and releases taking effect in each month."""
    hires = [0] * 12
    releases = [0] * 12
    for m in range(1, 12):
        delta = crews[m] - crews[m - 1]
        hires[m] = max(delta, 0)
        releases[m] = max(-delta, 0)
    return hires, releases


def simulate(dept, crews, demand, seasonal=None):
    p = PARAMS[dept]
    seasonal = seasonal or [0] * 12
    hires, releases = derive_moves(crews)

    backlog = []  # list of [age_in_months, count], oldest first
    rows = []
    for m in range(12):
        # 1) new requests join the queue
        backlog.append([0, int(demand[m])])
        waiting = sum(c for _, c in backlog)

        # 2) crews close requests, oldest first
        capacity = (crews[m] + seasonal[m]) * p["monthly_cap_per_crew"]
        remaining_cap = capacity
        closed_on_time = 0
        for item in backlog:
            take = min(item[1], remaining_cap)
            if item[0] < TARGET_MONTHS:
                closed_on_time += take
            item[1] -= take
            remaining_cap -= take
            if remaining_cap == 0:
                break
        backlog = [it for it in backlog if it[1] > 0]
        processed = capacity - remaining_cap

        # 3) month-end: age the queue and count new overdue events
        new_overdue = sum(c for a, c in backlog if a == TARGET_MONTHS - 1)
        new_overdue_30 = sum(c for a, c in backlog if a == TARGET_MONTHS)
        for it in backlog:
            it[0] += 1
        backlog_end = sum(c for _, c in backlog)

        # 4) costs
        training = hires[m + 1] if (m < 11 and PAY_DURING_TRAINING) else 0
        crew_cost = (crews[m] + training) * p["monthly_cost"]
        seasonal_cost = seasonal[m] * p["monthly_cost"] * SEASONAL_MULTIPLIER
        release_cost = releases[m] * p["annual_cost"] * RELEASE_COST_RATE
        overdue_cost = new_overdue * OVERDUE_COST + new_overdue_30 * OVERDUE_30_EXTRA
        idle_crews = remaining_cap / p["monthly_cap_per_crew"]

        rows.append({
            "department": dept,
            "month": MONTHS[m],
            "demand": int(demand[m]),
            "crews": crews[m],
            "seasonal": seasonal[m],
            "hires": hires[m],
            "releases": releases[m],
            "in_training": training,
            "capacity": capacity,
            "processed": processed,
            "backlog_end": backlog_end,
            "utilization": waiting / capacity if capacity else np.inf,
            "expected_ontime": closed_on_time / demand[m] if demand[m] else 1.0,
            "new_overdue": new_overdue,
            "new_overdue_30": new_overdue_30,
            "idle_crews": round(idle_crews, 2),
            "crew_cost": crew_cost,
            "seasonal_cost": seasonal_cost,
            "release_cost": release_cost,
            "overdue_cost": overdue_cost,
        })

    out = pd.DataFrame(rows)
    out["total_cost"] = out[["crew_cost", "seasonal_cost", "release_cost", "overdue_cost"]].sum(axis=1)
    return out


def summarize(sim):
    return pd.Series({
        "total_cost": sim["total_cost"].sum(),
        "staff_cost": sim[["crew_cost", "seasonal_cost", "release_cost"]].sum().sum(),
        "overdue_cost": sim["overdue_cost"].sum(),
        "ontime_rate": 1 - sim["new_overdue"].sum() / sim["demand"].sum(),
        "backlog_dec": sim["backlog_end"].iloc[-1],
        "idle_crew_months": sim["idle_crews"].sum(),
    })

**Sanity check.** One department, one month, numbers worked out by hand earlier: 50 Public Works crews (capacity 12,600) facing 10,080 requests should leave 10 idle crews and no overdue requests.

In [50]:
_check = simulate("PWDx", [50] * 12, [10_080] * 12)
assert _check.loc[0, "capacity"] == 12_600
assert _check.loc[0, "idle_crews"] == 10
assert _check["new_overdue"].sum() == 0
print("Sanity check passed:", _check.loc[0, ["capacity", "processed", "idle_crews", "crew_cost"]].to_dict())

Sanity check passed: {'capacity': 12600, 'processed': 10080, 'idle_crews': 10.0, 'crew_cost': 541666.6666666667}


## 5. Plan validation

Any plan must pass these checks before it is evaluated.

In [51]:
def validate_plan(dept, crews, seasonal=None):
    seasonal = seasonal or [0] * 12
    errors = []
    if len(crews) != 12 or len(seasonal) != 12:
        errors.append("crews and seasonal must have 12 values")
    if crews[0] != PARAMS[dept]["start_crews"]:
        errors.append(f"January crews must equal start level ({PARAMS[dept]['start_crews']})")
    if any((not float(c).is_integer()) or c < 0 for c in list(crews) + list(seasonal)):
        errors.append("crew counts must be non-negative integers")
    for m, s in enumerate(seasonal, start=1):
        if s > 0 and (dept not in SEASONAL_DEPTS or m not in SEASONAL_MONTHS):
            errors.append(f"seasonal crews not allowed for {dept} in month {m}")
    if errors:
        raise ValueError(f"{dept}: " + "; ".join(errors))
    return True

## 6. Runners

- `run_policy(policy, demand_df)`: builds the plan from `demand_df` and simulates it against the same demand.
- `evaluate_fixed_plan(policy, plan_demand_df, actual_demand_df)`: builds the plan from the forecast, then simulates it against a different actual demand. Used for sensitivity analysis and by the intake pipeline's shortage check.

In [52]:
def run_policy(policy, demand_df):
    sims = []
    for dept in PARAMS:
        demand = demand_df.loc[demand_df["department"] == dept].sort_values("month")["requests"].tolist()
        crews, seasonal = policy(dept, demand)
        validate_plan(dept, crews, seasonal)
        sims.append(simulate(dept, crews, demand, seasonal))
    return pd.concat(sims, ignore_index=True)


def evaluate_fixed_plan(policy, plan_demand_df, actual_demand_df):
    sims = []
    for dept in PARAMS:
        plan_dem = plan_demand_df.loc[plan_demand_df["department"] == dept].sort_values("month")["requests"].tolist()
        act_dem = actual_demand_df.loc[actual_demand_df["department"] == dept].sort_values("month")["requests"].tolist()
        crews, seasonal = policy(dept, plan_dem)
        validate_plan(dept, crews, seasonal)
        sims.append(simulate(dept, crews, act_dem, seasonal))
    return pd.concat(sims, ignore_index=True)

## 7. Staffing policy

| Decision | Rule | Why |
|---|---|---|
| Crews needed in a month | **(Forecast × (1 + buffer) + backlog carried in) ÷ capacity per crew**, rounded up | Unfinished work does not disappear, and the buffer protects against demand above forecast (with no buffer, on-time performance under +20% demand falls to 33%; with a 10% buffer it is 57%). |
| Excess crews after a peak | **Release only crews not needed again within the next `RELEASE_LOOKAHEAD` months** | Releasing costs 25% of annual pay = 3 months of pay, and re-hiring adds a paid training month. So if a crew is needed again within about 4 months, keeping it idle is cheaper than release-then-rehire. (Releasing right away instead would churn 29 hires / 36 releases in Public Works and cost more overall.) |
| Parks peak (April–October) | **Seasonal crews only**; no regular Parks hiring during the season | Regular crew for a *k*-month peak: \$40,000 + \$10,000·*k*. Seasonal crew: \$13,000·*k*. Break-even at *k* ≈ 13 months; the season is at most 7. |

Month-by-month procedure:

1. **January** stays at the starting level (assumption 6).
2. For each later month *m*, project the backlog carried in from the crews already planned.
3. **Required now** = ⌈(forecast[m] × (1 + buffer) + backlog) ÷ capacity⌉.
4. **Keep level** = the largest requirement in months *m … m + lookahead − 1* (future months use forecast only). Regular crews = max(required now, min(last month's crews, keep level)). So crews are hired when needed and released only if they will not be needed soon.
5. **Parks, April–October:** regular crews can only stay level or go down; seasonal crews fill any gap.
6. Hiring for month *m* is decided in *m−1* (paid training month); releases take effect after one month of notice. The evaluator charges both.

In [53]:
import math

BUFFER = 0.10            # buffer for need in the forecast
RELEASE_LOOKAHEAD = 4    # months; from the release-vs-keep break-even above


def staffing_policy(dept, demand, buffer=BUFFER, lookahead=RELEASE_LOOKAHEAD):
    p = PARAMS[dept]
    cap = p["monthly_cap_per_crew"]
    crews = [p["start_crews"]] + [0] * 11
    seasonal = [0] * 12
    future_req = [math.ceil(d * (1 + buffer) / cap) for d in demand]

    backlog = max(0, demand[0] - crews[0] * cap)  # carried into February
    for m in range(1, 12):
        required = math.ceil((demand[m] * (1 + buffer) + backlog) / cap)
        keep_level = max([required] + future_req[m + 1 : m + lookahead])
        regular = max(required, min(crews[m - 1], keep_level))

        in_season = dept in SEASONAL_DEPTS and (m + 1) in SEASONAL_MONTHS
        if in_season:
            crews[m] = min(crews[m - 1], regular)      # no regular hiring in season
            seasonal[m] = max(0, required - crews[m])  # seasonal crews fill the gap
        else:
            crews[m] = regular

        capacity = (crews[m] + seasonal[m]) * cap
        backlog = max(0, backlog + demand[m] - capacity)

    return crews, seasonal

## 8. 2025 staffing plan

Regular crews by month for each department, plus Parks seasonal crews, using `BUFFER` and `RELEASE_LOOKAHEAD` from Section 7.

In [54]:
policy_sim = run_policy(staffing_policy, demand_df)

plan_view = policy_sim.pivot(index="month", columns="department", values="crews")[list(PARAMS)]
plan_view["PARK_seasonal"] = policy_sim.loc[policy_sim["department"] == "PARK"].set_index("month")["seasonal"]
plan_view.index = plan_view.index.astype(str)
plan_view

department,PWDx,BTDT,PARK,ISD,PARK_seasonal
month,,,,,
2025-01,50,18,10,16,0
2025-02,50,23,10,22,0
2025-03,53,27,11,22,0
2025-04,56,27,11,22,2
2025-05,57,27,11,22,8
2025-06,57,27,11,24,14
2025-07,60,27,11,24,13
2025-08,72,27,11,27,10
2025-09,69,28,11,24,4


Hiring and release timing (a crew hired for month *m* trains in *m−1*; a crew released in month *m* was given notice in *m−1*).

In [55]:
moves = policy_sim.loc[(policy_sim["hires"] > 0) | (policy_sim["releases"] > 0),
                       ["department", "month", "crews", "hires", "releases"]]
moves.assign(month=moves["month"].astype(str)).reset_index(drop=True)

,department,month,crews,hires,releases
0,PWDx,2025-03,53,3,0
1,PWDx,2025-04,56,3,0
2,PWDx,2025-05,57,1,0
3,PWDx,2025-07,60,3,0
4,PWDx,2025-08,72,12,0
5,PWDx,2025-09,69,0,3
6,PWDx,2025-10,51,0,18
7,PWDx,2025-11,47,0,4
8,BTDT,2025-02,23,5,0
9,BTDT,2025-03,27,4,0


Annual results by department.

In [56]:
policy_summary = policy_sim.groupby("department").apply(summarize, include_groups=False)
policy_summary.loc["TOTAL"] = policy_summary.sum()
policy_summary.loc["TOTAL", "ontime_rate"] = 1 - policy_sim["new_overdue"].sum() / policy_sim["demand"].sum()

policy_summary.style.format({
    "total_cost": "${:,.0f}", "staff_cost": "${:,.0f}", "overdue_cost": "${:,.0f}",
    "ontime_rate": "{:.1%}", "backlog_dec": "{:,.0f}", "idle_crew_months": "{:.1f}",
})

,total_cost,staff_cost,overdue_cost,ontime_rate,backlog_dec,idle_crew_months
department,,,,,,
BTDT,"$3,059,880","$3,052,500","$7,380",99.6%,0,36.1
ISD,"$3,490,633","$3,488,333","$2,300",99.4%,0,36.2
PARK,"$2,006,000","$2,006,000",$0,100.0%,0,24.3
PWDx,"$8,298,333","$8,298,333",$0,100.0%,0,69.0
TOTAL,"$16,854,847","$16,845,167","$9,680",99.8%,0,165.7


## 9. Sensitivity: demand ±20%

The **plan is fixed** at what the policy chose for the base forecast; only actual demand changes. This shows how a plan built for the forecast holds up when the forecast is wrong. (Re-planning under each scenario is a separate question worth adding.)

In [57]:
scenario_df = (
    pd.read_csv("sensitivity_scenarios_2025.csv")
    .assign(month=lambda d: pd.to_datetime(d["month"]).dt.to_period("M"))
    .rename(columns={"scenario_requests": "requests"})
)
scenarios = ["Low (-20%)", "Baseline", "High (+20%)"]

rows = []
for label in scenarios:
    actual = scenario_df.loc[scenario_df["scenario"] == label, ["department", "month", "requests"]]
    sim = evaluate_fixed_plan(staffing_policy, demand_df, actual)
    for dept, s in sim.groupby("department").apply(summarize, include_groups=False).iterrows():
        rows.append({"scenario": label, "department": dept, **s.to_dict()})

sensitivity = pd.DataFrame(rows)
sensitivity.pivot(index="department", columns="scenario", values="total_cost")[scenarios].style.format("${:,.0f}")

scenario,Low (-20%),Baseline,High (+20%)
department,,,
BTDT,"$3,052,500","$3,059,880","$3,853,520"
ISD,"$3,488,333","$3,490,633","$3,576,373"
PARK,"$2,006,000","$2,006,000","$2,175,960"
PWDx,"$8,298,333","$8,298,333","$10,069,053"


In [58]:
sensitivity.pivot(index="department", columns="scenario", values="ontime_rate")[list(scenarios)].style.format("{:.1%}")

scenario,Low (-20%),Baseline,High (+20%)
department,,,
BTDT,100.0%,99.6%,60.6%
ISD,100.0%,99.4%,80.6%
PARK,100.0%,100.0%,65.6%
PWDx,100.0%,100.0%,51.5%


## 10. Lookup table for the intake pipeline

The final plan in one table, one row per department and month. The intake pipeline in Section 11 reads the same plan (`policy_sim`).

In [59]:
LOOKUP_COLS = ["department", "month", "crews", "seasonal", "hires", "releases",
               "capacity", "demand", "utilization", "expected_ontime", "total_cost"]

plan_table = policy_sim[LOOKUP_COLS].copy()
plan_table["month"] = plan_table["month"].astype(str)
plan_table.head(12)

,department,month,crews,seasonal,hires,releases,capacity,demand,utilization,expected_ontime,total_cost
0,PWDx,2025-01,50,0,0,0,12600,12437,0.987063,1.0,5.416667e+05
1,PWDx,2025-02,50,0,0,0,12600,10285,0.816270,1.0,5.741667e+05
2,PWDx,2025-03,53,0,3,0,13356,11995,0.898098,1.0,6.066667e+05
3,PWDx,2025-04,56,0,3,0,14112,12740,0.902778,1.0,6.175000e+05
4,PWDx,2025-05,57,0,1,0,14364,13031,0.907199,1.0,6.175000e+05
5,PWDx,2025-06,57,0,0,0,14364,12673,0.882275,1.0,6.500000e+05
6,PWDx,2025-07,60,0,3,0,15120,13678,0.904630,1.0,7.800000e+05
7,PWDx,2025-08,72,0,12,0,18144,16474,0.907959,1.0,7.800000e+05
8,PWDx,2025-09,69,0,0,3,17388,15662,0.900736,1.0,8.450000e+05
9,PWDx,2025-10,51,0,0,18,12852,11618,0.903984,1.0,1.137500e+06


## 11. Process Intake Pipeline

At the end of a month, the intake assistant's classified messages give an estimated request count per department (`estimated_requests` = classified messages × scale). This section compares that estimate with the forecast and checks whether the **fixed plan** can handle next month.

1. **Ratio** = estimated requests this month ÷ forecast this month.
2. **Next month's load** = next month's forecast × ratio (assumes the trend continues).
3. **Crews short** = ⌈(next month's load − next month's planned capacity) ÷ capacity per crew⌉.

Notes: the scale is calibrated so the normal stream reproduces the forecast exactly, so the normal file is a control case and should raise no alerts. Crews hired after a month-end alert start the month after next (one-month training), so the alert is an early warning; Parks can add seasonal crews immediately in April–October. Backlog carried over from the signal month is not added to next month's load.

In [60]:
import math

def check_next_month(intake_path, month="2025-08"):
    """End of `month`: compare estimated intake with the forecast and flag next month's shortage."""
    intake = pd.read_csv(intake_path)
    m = pd.Period(month, "M")
    nxt = m + 1
    rows = []
    for _, r in intake.iterrows():
        dept = r["department"]
        cap_per_crew = PARAMS[dept]["monthly_cap_per_crew"]
        forecast = demand_df[demand_df["department"] == dept].set_index("month")["requests"]
        plan = policy_sim[policy_sim["department"] == dept].set_index("month")

        ratio = r["estimated_requests"] / forecast[m]       # this month: actual vs. forecast
        next_load = forecast[nxt] * ratio                    # assume the trend continues
        next_cap = plan.loc[nxt, "capacity"]                 # fixed plan, unchanged
        short = max(0, math.ceil((next_load - next_cap) / cap_per_crew))

        rows.append({
            "department": dept, "observed_msgs": int(r["observed_count"]),
            "ratio": round(ratio, 2), "next_forecast": int(forecast[nxt]),
            "next_load": round(next_load), "next_capacity": int(next_cap),
            "utilization": next_load / next_cap, "crews_short": short,
        })
    return pd.DataFrame(rows)


def alert_messages(alerts, month="2025-08"):
    nxt = str(pd.Period(month, "M") + 1)
    for _, a in alerts.iterrows():
        if a["crews_short"] == 0:
            print(f"✅ {a['department']} {nxt}: OK (utilization {a['utilization']:.0%})")
        else:
            print(f"⚠️ {a['department']} {nxt}: {month} intake ran at {a['ratio']:.0%} of forecast. "
                  f"Expect {a['crews_short']} crews short (utilization {a['utilization']:.0%}). "
                  f"Decide hiring or overtime now.")

In [61]:
for path in ["extra_requests_august.csv", "extra_requests_august_parking_surge.csv"]:
    print(f"\n### {path}")
    alerts = check_next_month(path)
    alert_messages(alerts)
    display(alerts.style.format({"utilization": "{:.0%}"}))


### extra_requests_august.csv
✅ PWDx 2025-09: OK (utilization 90%)
✅ BTDT 2025-09: OK (utilization 91%)
✅ PARK 2025-09: OK (utilization 88%)
✅ ISD 2025-09: OK (utilization 89%)


,department,observed_msgs,ratio,next_forecast,next_load,next_capacity,utilization,crews_short
0,PWDx,49,1.000000,15662,15662,17388,90%,0
1,BTDT,18,1.000000,7990,7990,8820,91%,0
2,PARK,6,1.000000,1658,1658,1890,88%,0
3,ISD,7,1.000000,1804,1804,2016,89%,0



### extra_requests_august_parking_surge.csv
✅ PWDx 2025-09: OK (utilization 90%)
⚠️ BTDT 2025-09: 2025-08 intake ran at 183% of forecast. Expect 19 crews short (utilization 166%). Decide hiring or overtime now.
✅ PARK 2025-09: OK (utilization 88%)
✅ ISD 2025-09: OK (utilization 89%)


,department,observed_msgs,ratio,next_forecast,next_load,next_capacity,utilization,crews_short
0,PWDx,49,1.000000,15662,15662,17388,90%,0
1,BTDT,33,1.830000,7990,14648,8820,166%,19
2,PARK,6,1.000000,1658,1658,1890,88%,0
3,ISD,7,1.000000,1804,1804,2016,89%,0
